## Fetch Stimuli

In [2]:
import os
import numpy as np
import scipy.io
import h5py
from PIL import Image

import sys
sys.path.append("/home/ec2-user/Brain2VLM/codes/utils")
from nsd_access.nsda import NSDAccess

subject = "subj01"
imgidx_list = [0, 10, 25, 100]   # your imgidx values
outdir = "./org_images"
os.makedirs(outdir, exist_ok=True)

# Load NSD experiment design
nsd_expdesign = scipy.io.loadmat('../nsd/nsddata/experiments/nsd/nsd_expdesign.mat')
sharedix = nsd_expdesign['sharedix'] - 1

# Load NSD access
nsda = NSDAccess('../nsd/')
sf = h5py.File(nsda.stimuli_file, 'r')
sdataset = sf['imgBrick']

# Load stimulus mapping
stims_ave = np.load(f'../mrifeat/{subject}/{subject}_stims_ave.npy')

# Train/test split
tr_idx = np.zeros_like(stims_ave)
for i, s in enumerate(stims_ave):
    tr_idx[i] = 0 if s in sharedix else 1

# Save original images
for imgidx in imgidx_list:
    imgidx_te = np.where(tr_idx == 0)[0][imgidx]
    idx73k = stims_ave[imgidx_te]

    img = np.squeeze(sdataset[idx73k]).astype(np.uint8)
    Image.fromarray(img).save(f"{outdir}/{imgidx:05}_org.png")

print("Saved images.")

Saved images.


## Create Mosaic

In [ ]:
import os
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

# Prevent huge SVG size
# matplotlib.rcParams['svg.image_noscale'] = True

# ==============================
# CONFIG
# ==============================
SAMPLES_DIR = "../decoded/image-cvpr/subj01/samples"
selection_string = "95, 0, 8, 17, 18, 23, 35, 451, 777, 792"

OUTPUT_NAME = "outputs/reconstruction_results_cvpr"

# Max width for PNG to avoid draw.io crash
MAX_WIDTH_PX = 6000


# ==============================
# PARSE INPUT STRING
# ==============================
def parse_selection(sel_str):
    items = sel_str.split(",")
    parsed = []

    for item in items:
        item = item.strip()

        if "-" in item:
            idx, recon = item.split("-")
            parsed.append((int(idx), int(recon)))
        else:
            parsed.append((int(item), 4))  # default reconstruction

    return parsed


# ==============================
# CREATE PAPER FIGURE
# ==============================
def save_paper_figure(pairs):

    n = len(pairs)

    fig, axes = plt.subplots(
        2,
        n,
        figsize=(1.6 * n, 3.2),
        gridspec_kw={'wspace': 0.01, 'hspace': 0.01}
    )

    for col, (img_idx, recon_num) in enumerate(pairs):

        img_name = f"{img_idx:05d}"

        org_path = os.path.join(SAMPLES_DIR, f"{img_name}_org.png")
        recon_index = recon_num - 1
        recon_path = os.path.join(SAMPLES_DIR, f"{img_name}_{recon_index:03d}.png")

        org_img = Image.open(org_path)
        recon_img = Image.open(recon_path)

        # Row 1: Original image
        axes[0, col].imshow(org_img)
        axes[0, col].axis("off")

        # Row 2: Reconstruction
        axes[1, col].imshow(recon_img)
        axes[1, col].axis("off")

    plt.tight_layout(pad=0)

    # ==============================
    # SAVE SVG (vector editable)
    # ==============================
    plt.savefig(
        f"{OUTPUT_NAME}.svg",
        bbox_inches="tight",
        pad_inches=0,
        transparent=True
    )

    print(f"Saved:")
    print(f"  {OUTPUT_NAME}.svg")

    plt.close()


# ==============================
# RUN
# ==============================
parsed_pairs = parse_selection(selection_string)
save_paper_figure(parsed_pairs)

/tmp/ipykernel_553413/3757156847.py:73: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(pad=0)


Saved:
  reconstruction_results_cvpr.png
  reconstruction_results_cvpr.svg


**Examples:**
- `6` → Use image **00006** and display the **default reconstruction (4th sample)**.
- `6-2` → Use image **00006** and display the **2nd reconstruction sample**.